## EfficientNetB1 Classification

The super-resolved maize leaf images are classified into four Fall Armyworm-related classes using EfficientNetB1.

The model is trained using a two-stage transfer learning strategy. The first stage trains the classification head while keeping the pretrained backbone frozen, followed by fine-tuning the final backbone layers.

Training uses categorical cross-entropy with label smoothing, class weighting, AdamW optimization, learning-rate warmup with cosine decay, and early stopping.

In [ ]:
import os
import math
import random
from pathlib import Path

import numpy as np
import tensorflow as tf

from tensorflow.keras.applications import EfficientNetB1
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras import regularizers
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import Callback, EarlyStopping, ModelCheckpoint

from sklearn.metrics import classification_report, confusion_matrix


# =====================================================
# CONFIGURATION
# =====================================================
SEED = 42

DATASET_DIR = Path("path/to/super_resolved_dataset")

TRAIN_DIR = DATASET_DIR / "train"
VAL_DIR = DATASET_DIR / "val"
TEST_DIR = DATASET_DIR / "test"

OUTPUT_DIR = Path("path/to/output")

BATCH_SIZE = 32
IMAGE_SIZE = (240, 240)

NUM_CLASSES = 4

STAGE1_EPOCHS = 15
TOTAL_EPOCHS = 80

LR_STAGE1 = 2e-4
LR_STAGE2 = 1.8e-5

LR_MIN = 1e-7

WARMUP_STAGE1 = 3
WARMUP_STAGE2 = 2

WEIGHT_DECAY_STAGE1 = 1e-4
WEIGHT_DECAY_STAGE2 = 1e-5

LABEL_SMOOTHING = 0.04

DROPOUT_1 = 0.50
DROPOUT_2 = 0.32

FINE_TUNE_LAST_N_LAYERS = 60

CLASS_WEIGHTS = {
    0: 1.00,  # healthy
    1: 1.08,  # frass
    2: 1.35,  # egg
    3: 1.10   # larva
}

AUTOTUNE = tf.data.AUTOTUNE


# =====================================================
# REPRODUCIBILITY
# =====================================================
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)


# =====================================================
# LOAD DATASET
# =====================================================
train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=True,
    seed=SEED
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=False
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=False
)

class_names = train_dataset.class_names

print(f"Classes    : {class_names}")
print(f"Image size : {IMAGE_SIZE}")
print(f"Batch size : {BATCH_SIZE}")


# =====================================================
# DATA AUGMENTATION & PREPROCESSING
# =====================================================
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.022),
    tf.keras.layers.RandomZoom(0.055),
    tf.keras.layers.RandomTranslation(0.055, 0.055),
    tf.keras.layers.RandomContrast(0.025),
    tf.keras.layers.RandomBrightness(0.018)
])


def preprocess_train(images, labels):
    images = augmentation(
        images,
        training=True
    )

    images = preprocess_input(images)

    return images, labels


def preprocess_eval(images, labels):
    images = preprocess_input(images)

    return images, labels


training_data = (
    train_dataset
    .map(
        preprocess_train,
        num_parallel_calls=AUTOTUNE
    )
    .prefetch(AUTOTUNE)
)

validation_data = (
    val_dataset
    .map(
        preprocess_eval,
        num_parallel_calls=AUTOTUNE
    )
    .prefetch(AUTOTUNE)
)

test_data = (
    test_dataset
    .map(
        preprocess_eval,
        num_parallel_calls=AUTOTUNE
    )
    .prefetch(AUTOTUNE)
)


# =====================================================
# LEARNING-RATE SCHEDULER
# =====================================================
class WarmupCosineScheduler(Callback):

    def __init__(
        self,
        max_lr,
        min_lr,
        warmup_epochs,
        total_epochs
    ):
        super().__init__()

        self.max_lr = max_lr
        self.min_lr = min_lr
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs

    def on_epoch_begin(self, epoch, logs=None):

        if epoch < self.warmup_epochs:

            lr = (
                self.min_lr
                + (
                    self.max_lr - self.min_lr
                )
                * (epoch + 1)
                / self.warmup_epochs
            )

        else:

            progress = (
                epoch - self.warmup_epochs
            ) / max(
                1,
                self.total_epochs - self.warmup_epochs
            )

            lr = (
                self.min_lr
                + 0.5
                * (
                    self.max_lr - self.min_lr
                )
                * (
                    1 + math.cos(
                        math.pi * progress
                    )
                )
            )

        self.model.optimizer.learning_rate.assign(
            lr
        )


# =====================================================
# BUILD EFFICIENTNETB1
# =====================================================
backbone = EfficientNetB1(
    include_top=False,
    weights="imagenet",
    input_shape=(
        IMAGE_SIZE[0],
        IMAGE_SIZE[1],
        3
    ),
    pooling="avg"
)

backbone.trainable = False

model = Sequential(
    [
        backbone,

        Dense(
            512,
            activation="relu",
            kernel_regularizer=regularizers.l2(
                WEIGHT_DECAY_STAGE1
            )
        ),

        BatchNormalization(),

        Dropout(DROPOUT_1),

        Dense(
            256,
            activation="relu",
            kernel_regularizer=regularizers.l2(
                WEIGHT_DECAY_STAGE1
            )
        ),

        BatchNormalization(),

        Dropout(DROPOUT_2),

        Dense(
            NUM_CLASSES,
            activation="softmax"
        )
    ],
    name="EfficientNetB1_Classifier"
)


# =====================================================
# LOSS & OPTIMIZER
# =====================================================
loss_function = (
    tf.keras.losses.CategoricalCrossentropy(
        label_smoothing=LABEL_SMOOTHING
    )
)


# =====================================================
# STAGE 1 — TRAIN CLASSIFICATION HEAD
# =====================================================
model.compile(
    optimizer=AdamW(
        learning_rate=LR_STAGE1,
        weight_decay=WEIGHT_DECAY_STAGE1
    ),
    loss=loss_function,
    metrics=[
        "accuracy",
        Precision(name="precision"),
        Recall(name="recall")
    ]
)

stage1_callbacks = [
    WarmupCosineScheduler(
        max_lr=LR_STAGE1,
        min_lr=LR_MIN,
        warmup_epochs=WARMUP_STAGE1,
        total_epochs=STAGE1_EPOCHS
    ),

    EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    )
]

print("\n" + "=" * 55)
print("STAGE 1 — TRAIN CLASSIFICATION HEAD")
print("=" * 55)

history_stage1 = model.fit(
    training_data,
    validation_data=validation_data,
    epochs=STAGE1_EPOCHS,
    callbacks=stage1_callbacks,
    class_weight=CLASS_WEIGHTS
)


# =====================================================
# STAGE 2 — FINE-TUNING
# =====================================================
backbone.trainable = True

for layer in backbone.layers[
    :-FINE_TUNE_LAST_N_LAYERS
]:
    layer.trainable = False

for layer in backbone.layers:

    if isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    ):
        layer.trainable = False


model.compile(
    optimizer=AdamW(
        learning_rate=LR_STAGE2,
        weight_decay=WEIGHT_DECAY_STAGE2
    ),
    loss=loss_function,
    metrics=[
        "accuracy",
        Precision(name="precision"),
        Recall(name="recall")
    ]
)

stage2_callbacks = [
    WarmupCosineScheduler(
        max_lr=LR_STAGE2,
        min_lr=LR_MIN,
        warmup_epochs=WARMUP_STAGE2,
        total_epochs=(
            TOTAL_EPOCHS - STAGE1_EPOCHS
        )
    ),

    EarlyStopping(
        monitor="val_loss",
        patience=18,
        restore_best_weights=True
    )
]

print("\n" + "=" * 55)
print("STAGE 2 — FINE-TUNE EFFICIENTNETB1")
print("=" * 55)

history_stage2 = model.fit(
    training_data,
    validation_data=validation_data,
    initial_epoch=STAGE1_EPOCHS,
    epochs=TOTAL_EPOCHS,
    callbacks=stage2_callbacks,
    class_weight=CLASS_WEIGHTS
)


# =====================================================
# VALIDATION EVALUATION
# =====================================================
val_results = model.evaluate(
    validation_data,
    verbose=1
)

val_loss = val_results[0]
val_accuracy = val_results[1]
val_precision = val_results[2]
val_recall = val_results[3]

val_f1 = (
    2 * val_precision * val_recall
    / (
        val_precision
        + val_recall
        + 1e-7
    )
)


# =====================================================
# TEST EVALUATION
# =====================================================
test_results = model.evaluate(
    test_data,
    verbose=1
)

test_loss = test_results[0]
test_accuracy = test_results[1]
test_precision = test_results[2]
test_recall = test_results[3]

test_f1 = (
    2 * test_precision * test_recall
    / (
        test_precision
        + test_recall
        + 1e-7
    )
)


# =====================================================
# CLASSIFICATION REPORT
# =====================================================
y_true = []
y_pred = []

for images, labels in test_data:

    predictions = model.predict(
        images,
        verbose=0
    )

    y_true.extend(
        np.argmax(
            labels.numpy(),
            axis=1
        )
    )

    y_pred.extend(
        np.argmax(
            predictions,
            axis=1
        )
    )

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("\n=== CLASSIFICATION REPORT ===")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4
    )
)


# =====================================================
# CONFUSION MATRIX
# =====================================================
confusion = confusion_matrix(
    y_true,
    y_pred
)

print("\n=== CONFUSION MATRIX ===")
print(confusion)


# =====================================================
# SAVE MODEL
# =====================================================
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

model.save(
    OUTPUT_DIR / "efficientnetb1_best.keras"
)

print("\n=== FINAL RESULTS ===")
print(
    f"Validation Accuracy : "
    f"{val_accuracy * 100:.2f}%"
)

print(
    f"Validation F1       : "
    f"{val_f1 * 100:.2f}%"
)

print(
    f"Test Accuracy       : "
    f"{test_accuracy * 100:.2f}%"
)

print(
    f"Test F1             : "
    f"{test_f1 * 100:.2f}%"
)

print(
    f"\nModel saved to: "
    f"{OUTPUT_DIR / 'efficientnetb1_best.keras'}"
)

## ResNet50 Classification

ResNet50 is used as a comparative classification model to evaluate the performance of Fall Armyworm classification on super-resolved maize leaf images.

The model uses ImageNet pretrained weights and is trained through a two-stage transfer learning strategy. The classification head is trained first with the backbone frozen, followed by fine-tuning of the final backbone layers.

Categorical cross-entropy with label smoothing and AdamW optimization are used for training. Learning-rate decay is applied during the fine-tuning stage, while horizontal flipping, rotation, zoom, contrast, and brightness transformations are used as online data augmentation.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import tensorflow as tf

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras import regularizers
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras.optimizers import AdamW

from sklearn.metrics import classification_report, confusion_matrix


# =====================================================
# CONFIGURATION
# =====================================================
SEED = 42

DATASET_DIR = Path("path/to/super_resolved_dataset")

TRAIN_DIR = DATASET_DIR / "train"
VAL_DIR = DATASET_DIR / "val"
TEST_DIR = DATASET_DIR / "test"

OUTPUT_DIR = Path("path/to/output")

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

STAGE1_EPOCHS = 15
TOTAL_EPOCHS = 50

LR_STAGE1 = 8e-5
LR_STAGE2 = 5e-6
LR_STAGE2_MIN = 1e-6

WEIGHT_DECAY = 2e-4
L2_REGULARIZATION = 1e-4

LABEL_SMOOTHING = 0.04

DROPOUT_1 = 0.50
DROPOUT_2 = 0.40

FINE_TUNE_LAST_N_LAYERS = 15

AUTOTUNE = tf.data.AUTOTUNE


# =====================================================
# REPRODUCIBILITY
# =====================================================
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)


# =====================================================
# LOAD DATASET
# =====================================================
train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=True,
    seed=SEED
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=False
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=False
)

class_names = train_dataset.class_names
num_classes = len(class_names)

print(f"Classes    : {class_names}")
print(f"Image size : {IMAGE_SIZE}")
print(f"Batch size : {BATCH_SIZE}")


# =====================================================
# DATA AUGMENTATION & PREPROCESSING
# =====================================================
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.018),
    tf.keras.layers.RandomZoom(0.030),
    tf.keras.layers.RandomContrast(0.040),
    tf.keras.layers.RandomBrightness(0.030)
])


def preprocess_train(images, labels):
    images = augmentation(
        images,
        training=True
    )

    images = preprocess_input(images)

    return images, labels


def preprocess_eval(images, labels):
    images = preprocess_input(images)

    return images, labels


training_data = (
    train_dataset
    .map(
        preprocess_train,
        num_parallel_calls=AUTOTUNE
    )
    .prefetch(AUTOTUNE)
)

validation_data = (
    val_dataset
    .map(
        preprocess_eval,
        num_parallel_calls=AUTOTUNE
    )
    .prefetch(AUTOTUNE)
)

test_data = (
    test_dataset
    .map(
        preprocess_eval,
        num_parallel_calls=AUTOTUNE
    )
    .prefetch(AUTOTUNE)
)


# =====================================================
# BUILD RESNET50
# =====================================================
backbone = ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=(
        IMAGE_SIZE[0],
        IMAGE_SIZE[1],
        3
    ),
    pooling="avg"
)

backbone.trainable = False

model = Sequential(
    [
        backbone,

        Dense(
            128,
            activation="relu",
            kernel_regularizer=regularizers.l2(
                L2_REGULARIZATION
            )
        ),

        BatchNormalization(),

        Dropout(DROPOUT_1),

        Dense(
            64,
            activation="relu",
            kernel_regularizer=regularizers.l2(
                L2_REGULARIZATION
            )
        ),

        BatchNormalization(),

        Dropout(DROPOUT_2),

        Dense(
            num_classes,
            activation="softmax"
        )
    ],
    name="ResNet50_Classifier"
)


# =====================================================
# LOSS
# =====================================================
loss_function = (
    tf.keras.losses.CategoricalCrossentropy(
        label_smoothing=LABEL_SMOOTHING
    )
)


# =====================================================
# STAGE 1 — TRAIN CLASSIFICATION HEAD
# =====================================================
model.compile(
    optimizer=AdamW(
        learning_rate=LR_STAGE1,
        weight_decay=WEIGHT_DECAY,
        beta_1=0.9,
        beta_2=0.999,
        epsilon=1e-7
    ),
    loss=loss_function,
    metrics=[
        "accuracy",
        Precision(name="precision"),
        Recall(name="recall")
    ]
)

stage1_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=OUTPUT_DIR / "resnet50_stage1_best.keras",
        monitor="val_loss",
        mode="min",
        save_best_only=True
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=8,
        restore_best_weights=True
    )
]

print("\n" + "=" * 55)
print("STAGE 1 — TRAIN CLASSIFICATION HEAD")
print("=" * 55)

history_stage1 = model.fit(
    training_data,
    validation_data=validation_data,
    epochs=STAGE1_EPOCHS,
    callbacks=stage1_callbacks
)


# =====================================================
# STAGE 2 — FINE-TUNE LAST LAYERS
# =====================================================
backbone.trainable = True

for layer in backbone.layers[
    :-FINE_TUNE_LAST_N_LAYERS
]:
    layer.trainable = False

# Keep Batch Normalization layers frozen
for layer in backbone.layers:
    if isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    ):
        layer.trainable = False

trainable_layers = sum(
    layer.trainable
    for layer in backbone.layers
)

print(
    f"\nTrainable backbone layers: "
    f"{trainable_layers}"
)


# =====================================================
# COSINE DECAY FOR STAGE 2
# =====================================================
steps_per_epoch = (
    tf.data.experimental
    .cardinality(training_data)
    .numpy()
)

stage2_epochs = (
    TOTAL_EPOCHS - STAGE1_EPOCHS
)

decay_steps = (
    steps_per_epoch * stage2_epochs
)

lr_schedule = (
    tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=LR_STAGE2,
        decay_steps=decay_steps,
        alpha=LR_STAGE2_MIN / LR_STAGE2
    )
)


# =====================================================
# COMPILE STAGE 2
# =====================================================
model.compile(
    optimizer=AdamW(
        learning_rate=lr_schedule,
        weight_decay=WEIGHT_DECAY,
        beta_1=0.9,
        beta_2=0.999,
        epsilon=1e-7
    ),
    loss=loss_function,
    metrics=[
        "accuracy",
        Precision(name="precision"),
        Recall(name="recall")
    ]
)


stage2_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=OUTPUT_DIR / "resnet50_stage2_best.keras",
        monitor="val_loss",
        mode="min",
        save_best_only=True
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=18,
        restore_best_weights=True
    )
]


print("\n" + "=" * 55)
print("STAGE 2 — FINE-TUNING RESNET50")
print("=" * 55)

history_stage2 = model.fit(
    training_data,
    validation_data=validation_data,
    initial_epoch=STAGE1_EPOCHS,
    epochs=TOTAL_EPOCHS,
    callbacks=stage2_callbacks
)


# =====================================================
# EVALUATION
# =====================================================
print("\n" + "=" * 55)
print("FINAL EVALUATION")
print("=" * 55)

val_results = model.evaluate(
    validation_data,
    verbose=1
)

test_results = model.evaluate(
    test_data,
    verbose=1
)

val_loss = val_results[0]
val_accuracy = val_results[1]
val_precision = val_results[2]
val_recall = val_results[3]

test_loss = test_results[0]
test_accuracy = test_results[1]
test_precision = test_results[2]
test_recall = test_results[3]

val_f1 = (
    2 * val_precision * val_recall
    / (
        val_precision
        + val_recall
        + 1e-7
    )
)

test_f1 = (
    2 * test_precision * test_recall
    / (
        test_precision
        + test_recall
        + 1e-7
    )
)


# =====================================================
# CLASSIFICATION REPORT
# =====================================================
y_true = []
y_pred = []

for images, labels in test_data:

    predictions = model.predict(
        images,
        verbose=0
    )

    y_true.extend(
        np.argmax(
            labels.numpy(),
            axis=1
        )
    )

    y_pred.extend(
        np.argmax(
            predictions,
            axis=1
        )
    )

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("\n=== CLASSIFICATION REPORT ===")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4
    )
)


# =====================================================
# CONFUSION MATRIX
# =====================================================
confusion = confusion_matrix(
    y_true,
    y_pred
)

print("\n=== CONFUSION MATRIX ===")
print(confusion)


# =====================================================
# SAVE FINAL MODEL
# =====================================================
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

model.save(
    OUTPUT_DIR / "resnet50_best.keras"
)


# =====================================================
# FINAL SUMMARY
# =====================================================
print("\n" + "=" * 55)
print("RESNET50 CLASSIFICATION SUMMARY")
print("=" * 55)

print(
    f"Validation Accuracy : "
    f"{val_accuracy * 100:.2f}%"
)

print(
    f"Validation F1       : "
    f"{val_f1 * 100:.2f}%"
)

print(
    f"Test Accuracy       : "
    f"{test_accuracy * 100:.2f}%"
)

print(
    f"Test F1             : "
    f"{test_f1 * 100:.2f}%"
)

print(
    f"\nModel saved to: "
    f"{OUTPUT_DIR / 'resnet50_best.keras'}"
)

## MobileNetV3Large Classification

MobileNetV3Large is used as a comparative classification model for Fall Armyworm classification on super-resolved maize leaf images.

The model uses ImageNet pretrained weights and a two-stage transfer learning strategy. The classification head is trained first with the backbone frozen, followed by fine-tuning of the final backbone layers.

Training uses categorical cross-entropy with label smoothing, class weighting, AdamW optimization, learning-rate warmup with cosine decay in the first stage, and ReduceLROnPlateau during fine-tuning. Online data augmentation is applied to the training images.

In [ ]:
import os
import math
import random
from pathlib import Path

import numpy as np
import tensorflow as tf

from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras import regularizers
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import Callback

from sklearn.metrics import classification_report, confusion_matrix


# =====================================================
# CONFIGURATION
# =====================================================
SEED = 42

DATASET_DIR = Path("path/to/super_resolved_dataset")

TRAIN_DIR = DATASET_DIR / "train"
VAL_DIR = DATASET_DIR / "val"
TEST_DIR = DATASET_DIR / "test"

OUTPUT_DIR = Path("path/to/output")

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

STAGE1_EPOCHS = 15
TOTAL_EPOCHS = 80

LR_STAGE1 = 2e-4
LR_MIN_STAGE1 = 1e-7
WARMUP_STAGE1 = 3

LR_STAGE2 = 6e-6
LR_MIN_STAGE2 = 1e-7

WEIGHT_DECAY_STAGE1 = 1e-4
WEIGHT_DECAY_STAGE2 = 1e-5

LABEL_SMOOTHING = 0.02

DROPOUT_1 = 0.30
DROPOUT_2 = 0.18

FINE_TUNE_LAST_N_LAYERS = 60

CLASS_WEIGHTS = {
    0: 1.00,   # healthy
    1: 1.08,   # frass
    2: 1.28,   # egg
    3: 1.10    # larva
}

AUTOTUNE = tf.data.AUTOTUNE


# =====================================================
# REPRODUCIBILITY
# =====================================================
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)


# =====================================================
# LOAD DATASET
# =====================================================
train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=True,
    seed=SEED
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=False
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=False
)

class_names = train_dataset.class_names
num_classes = len(class_names)

print(f"Classes    : {class_names}")
print(f"Image size : {IMAGE_SIZE}")
print(f"Batch size : {BATCH_SIZE}")


# =====================================================
# DATA AUGMENTATION
# =====================================================
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.018),
    tf.keras.layers.RandomZoom(0.040),
    tf.keras.layers.RandomTranslation(
        0.040,
        0.040
    ),
    tf.keras.layers.RandomContrast(0.018),
    tf.keras.layers.RandomBrightness(0.010)
])


# =====================================================
# PREPROCESSING
# MobileNetV3Large uses built-in preprocessing
# when include_preprocessing=True.
# =====================================================
def preprocess_train(images, labels):
    images = augmentation(
        images,
        training=True
    )
    return images, labels


def preprocess_eval(images, labels):
    return images, labels


training_data = (
    train_dataset
    .map(
        preprocess_train,
        num_parallel_calls=AUTOTUNE
    )
    .prefetch(AUTOTUNE)
)

validation_data = (
    val_dataset
    .map(
        preprocess_eval,
        num_parallel_calls=AUTOTUNE
    )
    .prefetch(AUTOTUNE)
)

test_data = (
    test_dataset
    .map(
        preprocess_eval,
        num_parallel_calls=AUTOTUNE
    )
    .prefetch(AUTOTUNE)
)


# =====================================================
# LEARNING-RATE SCHEDULER
# =====================================================
class WarmupCosineScheduler(Callback):

    def __init__(
        self,
        max_lr,
        min_lr,
        warmup_epochs,
        total_epochs
    ):
        super().__init__()

        self.max_lr = max_lr
        self.min_lr = min_lr
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs

    def on_epoch_begin(self, epoch, logs=None):

        if epoch < self.warmup_epochs:

            lr = (
                self.min_lr
                + (
                    self.max_lr - self.min_lr
                )
                * (epoch + 1)
                / self.warmup_epochs
            )

        else:

            progress = (
                epoch - self.warmup_epochs
            ) / max(
                1,
                self.total_epochs - self.warmup_epochs
            )

            lr = (
                self.min_lr
                + 0.5
                * (
                    self.max_lr - self.min_lr
                )
                * (
                    1
                    + math.cos(
                        math.pi * progress
                    )
                )
            )

        self.model.optimizer.learning_rate.assign(lr)


# =====================================================
# VALIDATION F1 CHECKPOINT
# =====================================================
class ValidationF1Checkpoint(Callback):

    def __init__(
        self,
        filepath
    ):
        super().__init__()

        self.filepath = filepath
        self.best_f1 = -np.inf

    def on_epoch_end(
        self,
        epoch,
        logs=None
    ):

        logs = logs or {}

        precision = logs.get(
            "val_precision"
        )

        recall = logs.get(
            "val_recall"
        )

        if (
            precision is None
            or recall is None
        ):
            return

        val_f1 = (
            2
            * precision
            * recall
            / (
                precision
                + recall
                + 1e-7
            )
        )

        if val_f1 > self.best_f1:

            self.best_f1 = val_f1

            self.model.save(
                self.filepath
            )

            print(
                f"\nValidation F1 improved to "
                f"{val_f1:.4f}"
            )


# =====================================================
# BUILD MOBILENETV3LARGE
# =====================================================
backbone = MobileNetV3Large(
    include_top=False,
    weights="imagenet",
    input_shape=(
        IMAGE_SIZE[0],
        IMAGE_SIZE[1],
        3
    ),
    pooling="avg",
    include_preprocessing=True
)

backbone.trainable = False

model = Sequential(
    [
        backbone,

        Dense(
            512,
            activation="relu",
            kernel_regularizer=regularizers.l2(
                WEIGHT_DECAY_STAGE1
            )
        ),

        BatchNormalization(),

        Dropout(DROPOUT_1),

        Dense(
            256,
            activation="relu",
            kernel_regularizer=regularizers.l2(
                WEIGHT_DECAY_STAGE1
            )
        ),

        BatchNormalization(),

        Dropout(DROPOUT_2),

        Dense(
            num_classes,
            activation="softmax"
        )
    ],
    name="MobileNetV3Large_Classifier"
)


# =====================================================
# LOSS
# =====================================================
loss_function = (
    tf.keras.losses.CategoricalCrossentropy(
        label_smoothing=LABEL_SMOOTHING
    )
)


# =====================================================
# OUTPUT PATHS
# =====================================================
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

stage1_loss_path = (
    OUTPUT_DIR /
    "mobilenetv3large_stage1_val_loss.keras"
)

stage2_loss_path = (
    OUTPUT_DIR /
    "mobilenetv3large_stage2_val_loss.keras"
)

stage1_f1_path = (
    OUTPUT_DIR /
    "mobilenetv3large_stage1_val_f1.keras"
)

stage2_f1_path = (
    OUTPUT_DIR /
    "mobilenetv3large_stage2_val_f1.keras"
)

final_model_path = (
    OUTPUT_DIR /
    "mobilenetv3large_best.keras"
)


# =====================================================
# STAGE 1 — TRAIN CLASSIFICATION HEAD
# =====================================================
model.compile(
    optimizer=AdamW(
        learning_rate=LR_STAGE1,
        weight_decay=WEIGHT_DECAY_STAGE1
    ),
    loss=loss_function,
    metrics=[
        "accuracy",
        Precision(name="precision"),
        Recall(name="recall")
    ]
)

stage1_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=stage1_loss_path,
        monitor="val_loss",
        mode="min",
        save_best_only=True
    ),

    ValidationF1Checkpoint(
        filepath=stage1_f1_path
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=8,
        restore_best_weights=True
    ),

    WarmupCosineScheduler(
        max_lr=LR_STAGE1,
        min_lr=LR_MIN_STAGE1,
        warmup_epochs=WARMUP_STAGE1,
        total_epochs=STAGE1_EPOCHS
    )
]

print("\n" + "=" * 55)
print("STAGE 1 — TRAIN CLASSIFICATION HEAD")
print("=" * 55)

history_stage1 = model.fit(
    training_data,
    validation_data=validation_data,
    epochs=STAGE1_EPOCHS,
    callbacks=stage1_callbacks,
    class_weight=CLASS_WEIGHTS
)


# =====================================================
# STAGE 2 — FINE-TUNING LAST 60 LAYERS
# =====================================================
backbone.trainable = True

for layer in backbone.layers[
    :-FINE_TUNE_LAST_N_LAYERS
]:
    layer.trainable = False

# Keep Batch Normalization layers frozen
for layer in backbone.layers:

    if isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    ):
        layer.trainable = False

trainable_layers = sum(
    layer.trainable
    for layer in backbone.layers
)

print(
    f"\nTrainable backbone layers: "
    f"{trainable_layers}"
)


# =====================================================
# STAGE 2 OPTIMIZER
# =====================================================
model.compile(
    optimizer=AdamW(
        learning_rate=LR_STAGE2,
        weight_decay=WEIGHT_DECAY_STAGE2
    ),
    loss=loss_function,
    metrics=[
        "accuracy",
        Precision(name="precision"),
        Recall(name="recall")
    ]
)


# =====================================================
# STAGE 2 CALLBACKS
# =====================================================
stage2_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=stage2_loss_path,
        monitor="val_loss",
        mode="min",
        save_best_only=True
    ),

    ValidationF1Checkpoint(
        filepath=stage2_f1_path
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=7,
        min_lr=LR_MIN_STAGE2,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=18,
        restore_best_weights=True
    )
]


# =====================================================
# TRAIN STAGE 2
# =====================================================
print("\n" + "=" * 55)
print("STAGE 2 — FINE-TUNING MOBILENETV3LARGE")
print("=" * 55)

history_stage2 = model.fit(
    training_data,
    validation_data=validation_data,
    initial_epoch=STAGE1_EPOCHS,
    epochs=TOTAL_EPOCHS,
    callbacks=stage2_callbacks,
    class_weight=CLASS_WEIGHTS
)


# =====================================================
# SELECT FINAL MODEL USING VALIDATION F1
# =====================================================
def compile_for_evaluation(model):
    model.compile(
        optimizer=AdamW(
            learning_rate=LR_STAGE2,
            weight_decay=WEIGHT_DECAY_STAGE2
        ),
        loss=loss_function,
        metrics=[
            "accuracy",
            Precision(name="precision"),
            Recall(name="recall")
        ]
    )

    return model


candidate_paths = {
    "stage1_val_loss": stage1_loss_path,
    "stage1_val_f1": stage1_f1_path,
    "stage2_val_loss": stage2_loss_path,
    "stage2_val_f1": stage2_f1_path
}

candidate_results = {}

for name, path in candidate_paths.items():

    if not path.exists():
        continue

    candidate_model = compile_for_evaluation(
        tf.keras.models.load_model(
            path,
            compile=False
        )
    )

    loss, accuracy, precision, recall = (
        candidate_model.evaluate(
            validation_data,
            verbose=0
        )
    )

    f1 = (
        2
        * precision
        * recall
        / (
            precision
            + recall
            + 1e-7
        )
    )

    candidate_results[name] = {
        "path": path,
        "loss": float(loss),
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1)
    }

    print(
        f"{name:<16} | "
        f"val_loss={loss:.4f} | "
        f"val_acc={accuracy:.4f} | "
        f"val_f1={f1:.4f}"
    )


best_candidate = max(
    candidate_results,
    key=lambda name: (
        candidate_results[name]["f1"],
        -candidate_results[name]["loss"]
    )
)

final_model = compile_for_evaluation(
    tf.keras.models.load_model(
        candidate_results[best_candidate]["path"],
        compile=False
    )
)

final_model.save(
    final_model_path
)

print(
    f"\nFinal model selected from: "
    f"{best_candidate}"
)


# =====================================================
# VALIDATION EVALUATION
# =====================================================
val_results = final_model.evaluate(
    validation_data,
    verbose=1
)

val_loss = val_results[0]
val_accuracy = val_results[1]
val_precision = val_results[2]
val_recall = val_results[3]

val_f1 = (
    2
    * val_precision
    * val_recall
    / (
        val_precision
        + val_recall
        + 1e-7
    )
)


# =====================================================
# TEST EVALUATION
# =====================================================
test_results = final_model.evaluate(
    test_data,
    verbose=1
)

test_loss = test_results[0]
test_accuracy = test_results[1]
test_precision = test_results[2]
test_recall = test_results[3]

test_f1 = (
    2
    * test_precision
    * test_recall
    / (
        test_precision
        + test_recall
        + 1e-7
    )
)


# =====================================================
# CLASSIFICATION REPORT
# =====================================================
y_true = []
y_pred = []

for images, labels in test_data:

    predictions = final_model.predict(
        images,
        verbose=0
    )

    y_true.extend(
        np.argmax(
            labels.numpy(),
            axis=1
        )
    )

    y_pred.extend(
        np.argmax(
            predictions,
            axis=1
        )
    )

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("\n=== CLASSIFICATION REPORT ===")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4
    )
)


# =====================================================
# CONFUSION MATRIX
# =====================================================
confusion = confusion_matrix(
    y_true,
    y_pred
)

print("\n=== CONFUSION MATRIX ===")
print(confusion)


# =====================================================
# FINAL SUMMARY
# =====================================================
print("\n" + "=" * 55)
print("MOBILENETV3LARGE CLASSIFICATION SUMMARY")
print("=" * 55)

print(
    f"Validation Accuracy : "
    f"{val_accuracy * 100:.2f}%"
)

print(
    f"Validation F1       : "
    f"{val_f1 * 100:.2f}%"
)

print(
    f"Test Accuracy       : "
    f"{test_accuracy * 100:.2f}%"
)

print(
    f"Test F1             : "
    f"{test_f1 * 100:.2f}%"
)

print(
    f"Final model         : "
    f"{best_candidate}"
)

print(
    f"Model saved to      : "
    f"{final_model_path}"
)